This notebook shows simple ways to handle datasets with the huggingface **datasets** library

In [ ]:
from torch.utils.data import DataLoader

from datasets import load_dataset
from src.utils import paths

dataset_train = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
dataloader = DataLoader(dataset_train)

#downlaod WMT 2014 English-German
# dataset_train = load_dataset("wmt14", "fr-en", split="train")
# dataloader = DataLoader(dataset_train)
len(dataset_train)

In [ ]:
from datasets import load_dataset

# Load Croatian Wikipedia (replace the date with your target version)
dataset = load_dataset("wikimedia/wikipedia", "20231101.hr", 
    cache_dir=paths.DATA_DIR / "wikimedia/wikipedia/20231101/hr/")
print(len(dataset["train"]))
print(len(dataset["train"][2]["text"]))
print(len(dataset["train"][4]["text"]))
print(len(dataset["train"][23]["text"]))
print(len(dataset["train"][1]["text"]))

In [ ]:
dataset["train"][6]

In [ ]:
print(dataset_train)
x = dataset_train[:]
len(" ".join(x["text"]))

In [ ]:
count = 5
for i in dataloader:
    print(i["text"])
    count -= 1
    if count == 0:
        break

In [ ]:
s = "String        example   "
" ".join(s.split())

In [ ]:
from datasets import load_dataset

hf_dataset = load_dataset(
    "sentence-transformers/parallel-sentences-jw300", 
    "en-hr", 
    cache_dir=paths.DATA_DIR / "sentence-transformers/parallel-sentences-jw300",
    split="train")


hf_dataset[1000000]

In [ ]:
hf_dataset["train"][0]

In [ ]:
dataset = load_dataset("bentrevett/multi30k", split="train")#, data_dir="data/multi30k_de_en")
dataset[0]

In [ ]:
ds = load_dataset("Helsinki-NLP/opus-100", "en-hr", cache_dir=paths.DATA_DIR)
len(ds['train'])

In [ ]:
from datasets import load_dataset
from src.utils import paths

hf_dataset = load_dataset(
    "hrwac", 
    cache_dir=paths.DATA_DIR / "hrwac",
    )


hf_dataset[1000000]

In [ ]:
import gzip
from urllib.request import urlretrieve

from src.utils import paths

OUT = paths.DATA_DIR / "hrwac"
OUT.mkdir(parents=True, exist_ok=True)

URLS = [
    f"https://www.clarin.si/repository/xmlui/bitstream/handle/11356/1064/hrWaC2.1.{i:02d}.xml.gz"
    for i in range(1, 15)
]

# 1. Download (resume-safe: skip if file exists with non-zero size)
for url in URLS:
    fname = OUT / url.rsplit("/", 1)[-1]
    if fname.exists() and fname.stat().st_size > 0:
        print(f"skip {fname.name}")
        continue
    print(f"downloading {fname.name}")
    urlretrieve(url, fname)

# 2. Parse vertical XML and reconstruct sentences (plain text per line)
def iter_sentences(gz_path):
    """
    hrWaC vertical format:
      <text id="..." url="..." ...>
        <p>
          <s>
            token<TAB>msd<TAB>lemma
            token<TAB>msd<TAB>lemma
            ...
          </s>
        </p>
      </text>
    Yields one detokenized sentence string at a time.
    """
    tokens = []
    in_sentence = False
    with gzip.open(gz_path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("<s"):
                tokens = []
                in_sentence = True
            elif line.startswith("</s>"):
                if tokens:
                    yield detokenize(tokens)
                in_sentence = False
            elif in_sentence and line and not line.startswith("<"):
                # token<TAB>msd<TAB>lemma  — keep only the surface form
                surface = line.split("\t", 1)[0]
                tokens.append(surface)

def detokenize(tokens):
    # Minimal Croatian detokenization: don't put a space before
    # closing punctuation, do put a space everywhere else.
    out = []
    no_space_before = {".", ",", ";", ":", "!", "?", ")", "]", "}", "”", "»", "…"}
    no_space_after  = {"(", "[", "{", "„", "«"}
    for i, t in enumerate(tokens):
        if i > 0 and t not in no_space_before and tokens[i-1] not in no_space_after:
            out.append(" ")
        out.append(t)
    return "".join(out)

# 3. Write to a single newline-delimited plaintext file (or shard as you prefer)
with open(OUT / "hrwac_sentences.txt", "w", encoding="utf-8") as out:
    for gz in sorted(OUT.glob("hrWaC2.1.*.xml.gz")):
        print(f"parsing {gz.name}")
        for sent in iter_sentences(gz):
            out.write(sent + "\n")

Ljubešić, Nikola; Esplà-Gomis, Miquel; Ortiz Rojas, Sergio; Klubička, Filip and Toral, Antonio, 2016, Croatian-English parallel corpus hrenWaC 2.0, Slovenian language resource repository CLARIN.SI, ISSN 2820-4042,

In [ ]:
import datasets

riznica = datasets.load_dataset("classla/xlm-r-bertic-data", split="riznica", streaming=True)

In [ ]:
from datasets import load_dataset
from src.utils import paths

arc_hr   = load_dataset("alexandrainst/m_arc", "hr", cache_dir=paths.DATA_DIR)
hella_hr = load_dataset("alexandrainst/m_hellaswag", "hr", cache_dir=paths.DATA_DIR)
mmlu_hr  = load_dataset("alexandrainst/m_mmlu", "hr", cache_dir=paths.DATA_DIR)

In [ ]:
arc_hr["train"][5]["instruction"]

In [ ]:
from datasets import load_dataset

# ARC — note the two configs; Okapi uses the Challenge split
arc      = load_dataset("allenai/ai2_arc", "ARC-Challenge", cache_dir=paths.DATA_DIR)
arc_easy = load_dataset("allenai/ai2_arc", "ARC-Easy", cache_dir=paths.DATA_DIR)

# HellaSwag
hella    = load_dataset("Rowan/hellaswag", cache_dir=paths.DATA_DIR)

# MMLU — "all" gives every subject; or pass a single subject name
mmlu     = load_dataset("cais/mmlu", "all", cache_dir=paths.DATA_DIR)

In [ ]:
from datasets import load_dataset
from src.utils import paths

ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", cache_dir=paths.DATA_DIR / "fineweb-edu")

In [ ]:
ds["train"][0]

In [ ]:
from tqdm import tqdm

for data in tqdm(ds["train"]):
    pass